# Phase III Reynolds Solutions: Re = 500 and 1000 (Standard Coupled SGS)

This notebook is a copy of the trimmed Reynolds-number notebook, but it keeps the solver on the standard coupled SGS path instead of switching to `fractional_step`.

It runs the report cases:

| Reynolds number | Nodes | Relative mesh spacing label |
|---:|---:|---:|
| 500 | `33 x 33` | `h = 4` |
| 1000 | `65 x 65` | `h = 2` |

The solver is patched in memory, so `main_solver.py` is not modified. Results are written to a separate output folder so they do not reuse fractional-step or accelerated cached data.

In [1]:
from pathlib import Path
import contextlib
import io
import json
import os
import shutil
import time

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

if Path.cwd().name != "start-code" and (Path.cwd() / "start-code").exists():
    os.chdir(Path.cwd() / "start-code")

plot_dir = Path("Phase III Reynolds Re500 Re1000 Coupled Standard")
plot_dir.mkdir(exist_ok=True)

cache_dir = plot_dir / "case_cache"
cache_dir.mkdir(exist_ok=True)

print(f"Working directory: {Path.cwd()}")
print(f"Plot output directory: {plot_dir.resolve()}")
print(f"Cache directory: {cache_dir.resolve()}")

Working directory: /Users/windy/mae4100-courseproject2/start-code
Plot output directory: /Users/windy/mae4100-courseproject2/start-code/Phase III Reynolds Re500 Re1000 Coupled Standard
Cache directory: /Users/windy/mae4100-courseproject2/start-code/Phase III Reynolds Re500 Re1000 Coupled Standard/case_cache


In [2]:
REYNOLDS_CASES = [
    {"Re": 500.0, "nodes": 33, "h_label": 4, "cfl": 0.20},
    {"Re": 1000.0, "nodes": 65, "h_label": 2, "cfl": 0.10},
]

# Keep cached results once they have been generated. Set FORCE_RERUN = True to recompute both cases.
USE_CACHE = True
FORCE_RERUN = False

pd.DataFrame(REYNOLDS_CASES)

,Re,nodes,h_label,cfl
0,500.0,33,4,0.2
1,1000.0,65,2,0.1


In [3]:
MAIN_PLOT_FILES = [
    "ucontour.png",
    "vcontour.png",
    "pcontour.png",
    "residualcomponent.png",
    "residual.png",
]


def case_label(case):
    return f"Re{int(case['Re']):04d}_{case['nodes']}x{case['nodes']}"


def format_solver_float(value):
    return f"{value:<16g}"


def apply_replacements(source, replacements, solver_path):
    for old, new in replacements.items():
        if old not in source:
            raise RuntimeError(f"Could not find expected setting in {solver_path}: {old}")
        source = source.replace(old, new, 1)
    return source


def patched_solver_source(case):
    solver_path = Path("main_solver.py")
    source = solver_path.read_text()
    nodes = case["nodes"]

    replacements = {
        "imax = 9               # Number of points in the x-direction (use odd numbers only)":
            f"imax = {nodes:<15}# Number of points in the x-direction (use odd numbers only)",
        "jmax = 9               # Number of points in the y-direction (use odd numbers only)":
            f"jmax = {nodes:<15}# Number of points in the y-direction (use odd numbers only)",
        "cfl = 0.5              # CFL number used to determine time step":
            f"cfl = {format_solver_float(case['cfl'])}# CFL number used to determine time step",
        "Re = 10.0              # Reynolds number = rho*Uinf*L/rmu":
            f"Re = {format_solver_float(case['Re'])}# Reynolds number = rho*Uinf*L/rmu",
        "iterout = 5000         # Number of time steps between solution output":
            "iterout = 100000000    # Number of time steps between solution output",
        "vectorize = False":
            "vectorize = True",
    }
    return solver_path, apply_replacements(source, replacements, solver_path)


def archive_solver_plots(label):
    case_plot_dir = plot_dir / label
    case_plot_dir.mkdir(exist_ok=True)
    for filename in MAIN_PLOT_FILES:
        src = Path(filename)
        if src.exists():
            dst = case_plot_dir / filename
            if dst.exists():
                dst.unlink()
            shutil.move(str(src), str(dst))


def save_case_cache(case, result):
    label = case_label(case)
    np.savez_compressed(
        cache_dir / f"{label}.npz",
        u=result["u"],
        convVector=result["convVector"],
        res=result["res"],
    )
    metadata = {k: v for k, v in result.items() if k not in {"u", "convVector", "res", "captured_output"}}
    metadata["captured_output_tail"] = result["captured_output"].splitlines()[-12:]
    (cache_dir / f"{label}.json").write_text(json.dumps(metadata, indent=2))


def load_case_cache(case):
    label = case_label(case)
    npz_path = cache_dir / f"{label}.npz"
    json_path = cache_dir / f"{label}.json"
    if not npz_path.exists() or not json_path.exists():
        return None

    arrays = np.load(npz_path)
    metadata = json.loads(json_path.read_text())
    metadata["u"] = arrays["u"]
    metadata["convVector"] = arrays["convVector"]
    metadata["res"] = arrays["res"]
    metadata["captured_output"] = "\n".join(metadata.pop("captured_output_tail", []))
    return metadata


def run_solver_case(case):
    label = case_label(case)
    cached = load_case_cache(case) if USE_CACHE and not FORCE_RERUN else None
    if cached is not None:
        print(f"Loaded cached result for {label}")
        return cached

    solver_path, source = patched_solver_source(case)
    namespace = {
        "__file__": str(solver_path.resolve()),
        "__name__": "__main__",
    }

    print(
        f"Running coupled SGS {label}: "
        f"Re = {case['Re']:g}, nodes = {case['nodes']}x{case['nodes']}, "
        f"h = {case['h_label']}, CFL = {case['cfl']}"
    )
    start_time = time.perf_counter()
    with contextlib.redirect_stdout(io.StringIO()) as captured_output:
        exec(compile(source, str(solver_path), "exec"), namespace)
    elapsed_time = time.perf_counter() - start_time
    archive_solver_plots(label)
    plt.close("all")

    u = namespace["u"].copy()
    imax, jmax, _ = u.shape
    result = {
        "label": label,
        "solver_method": namespace["solver_method"],
        "Re": case["Re"],
        "nodes": case["nodes"],
        "h_label": case["h_label"],
        "cfl": case["cfl"],
        "dx": (namespace["xmax"] - namespace["xmin"]) / (imax - 1),
        "dy": (namespace["ymax"] - namespace["ymin"]) / (jmax - 1),
        "viscosity": float(namespace["rmu"]),
        "iterations": int(namespace["n"]),
        "converged": bool(namespace["isConverged"]),
        "final_conv": float(namespace["conv"]),
        "elapsed_time_sec": float(elapsed_time),
        "xmin": float(namespace["xmin"]),
        "xmax": float(namespace["xmax"]),
        "ymin": float(namespace["ymin"]),
        "ymax": float(namespace["ymax"]),
        "u": u,
        "res": np.array(namespace["res"], copy=True),
        "convVector": np.array(namespace["convVector"], copy=True),
        "captured_output": captured_output.getvalue(),
    }
    save_case_cache(case, result)
    return result


def build_summary(results):
    return pd.DataFrame([
        {
            "Re": result["Re"],
            "nodes": f"{result['nodes']}x{result['nodes']}",
            "solver method": result["solver_method"],
            "h label": result["h_label"],
            "dx = dy (m)": result["dx"],
            "CFL": result["cfl"],
            "viscosity (N*s/m^2)": result["viscosity"],
            "iterations": result["iterations"],
            "converged": result["converged"],
            "final conv": result["final_conv"],
            "wall time (s)": result["elapsed_time_sec"],
        }
        for result in results
    ])


def write_partial_summary(results):
    summary = build_summary(results)
    summary.to_csv(plot_dir / "summary_partial.csv", index=False)
    summary.to_csv(cache_dir / "summary_partial.csv", index=False)
    return summary

In [ ]:
results = []
for case in REYNOLDS_CASES:
    result = run_solver_case(case)
    results.append(result)
    summary = write_partial_summary(results)
    print(f"Saved partial summary after {len(results)} completed case(s): {plot_dir / 'summary_partial.csv'}")

summary

Running coupled SGS Re0500_33x33: Re = 500, nodes = 33x33, h = 4, CFL = 0.2
Saved partial summary after 1 completed case(s): Phase III Reynolds Re500 Re1000 Coupled Standard/summary_partial.csv
Running coupled SGS Re1000_65x65: Re = 1000, nodes = 65x65, h = 2, CFL = 0.1


In [ ]:
summary_csv = plot_dir / "summary_partial.csv"
if not summary_csv.exists():
    raise FileNotFoundError("No partial summary found yet. Run the case loop until at least one case finishes.")

print(f"Loaded partial summary from: {summary_csv.resolve()}")
summary = pd.read_csv(summary_csv)

print("\nCase runtimes and convergence available so far:")
display(summary)

In [ ]:
def normalized_coordinates(result):
    x = np.linspace(result["xmin"], result["xmax"], result["nodes"])
    y = np.linspace(result["ymin"], result["ymax"], result["nodes"])
    length = result["xmax"] - result["xmin"]
    return x / length, y / length

In [ ]:
fontsize = 12

plt.figure(figsize=(7, 5))
for result in results:
    _, y_norm = normalized_coordinates(result)
    u = result["u"]
    i_mid = (result["nodes"] - 1) // 2
    plt.plot(u[i_mid, :, 1], y_norm, linewidth=2, label=f"Re={int(result['Re'])}")
plt.xlabel("u velocity (m/s)", fontsize=fontsize)
plt.ylabel("y / L", fontsize=fontsize)
plt.title("Vertical centerline u-velocity", fontsize=fontsize)
plt.legend(loc="best", fontsize=fontsize, frameon=False)
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig(plot_dir / "vertical_centerline_u_Re500_Re1000.png", dpi=300)
plt.show()

plt.figure(figsize=(7, 5))
for result in results:
    x_norm, _ = normalized_coordinates(result)
    u = result["u"]
    j_mid = (result["nodes"] - 1) // 2
    plt.plot(x_norm, u[:, j_mid, 2], linewidth=2, label=f"Re={int(result['Re'])}")
plt.xlabel("x / L", fontsize=fontsize)
plt.ylabel("v velocity (m/s)", fontsize=fontsize)
plt.title("Horizontal centerline v-velocity", fontsize=fontsize)
plt.legend(loc="best", fontsize=fontsize, frameon=False)
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig(plot_dir / "horizontal_centerline_v_Re500_Re1000.png", dpi=300)
plt.show()

plt.figure(figsize=(7, 5))
for result in results:
    _, y_norm = normalized_coordinates(result)
    u = result["u"]
    i_mid = (result["nodes"] - 1) // 2
    plt.plot(u[i_mid, :, 0], y_norm, linewidth=2, label=f"Re={int(result['Re'])}")
plt.xlabel("p (N/m^2)", fontsize=fontsize)
plt.ylabel("y / L", fontsize=fontsize)
plt.title("Vertical centerline pressure", fontsize=fontsize)
plt.legend(loc="best", fontsize=fontsize, frameon=False)
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig(plot_dir / "vertical_centerline_pressure_Re500_Re1000.png", dpi=300)
plt.show()

In [ ]:
def contour_grid(result):
    x_norm, y_norm = normalized_coordinates(result)
    return np.meshgrid(x_norm, y_norm, indexing="ij")

fig, axes = plt.subplots(len(results), 3, figsize=(13, 3.2 * len(results)), constrained_layout=True)

for row, result in enumerate(results):
    X, Y = contour_grid(result)
    u = result["u"]
    fields = [
        (u[:, :, 1], "u (m/s)"),
        (u[:, :, 2], "v (m/s)"),
        (u[:, :, 0], "p (N/m^2)"),
    ]
    for col, (field, title) in enumerate(fields):
        ax = axes[row, col]
        contour = ax.contourf(X, Y, field, levels=20)
        fig.colorbar(contour, ax=ax)
        ax.set_aspect("equal")
        ax.set_xlabel("x / L")
        ax.set_ylabel("y / L")
        ax.set_title(f"Re={int(result['Re'])}, {result['nodes']}x{result['nodes']} {title}")

plt.savefig(plot_dir / "reynolds_contour_comparison_Re500_Re1000.png", dpi=300)
plt.show()

In [ ]:
analysis_rows = []
for result in results:
    u = result["u"]
    analysis_rows.append({
        "Re": result["Re"],
        "nodes": f"{result['nodes']}x{result['nodes']}",
        "solver_method": result["solver_method"],
        "u_min": np.min(u[:, :, 1]),
        "u_max": np.max(u[:, :, 1]),
        "v_min": np.min(u[:, :, 2]),
        "v_max": np.max(u[:, :, 2]),
        "p_min": np.min(u[:, :, 0]),
        "p_max": np.max(u[:, :, 0]),
    })

flow_metrics = pd.DataFrame(analysis_rows)
flow_metrics